# 01 — Exploratory Data Analysis | Ames Housing

**Goal:** understand the target, feature distributions, missingness, and potential data-quality issues without modifying the source dataset. Run from the repository root or `notebooks/`.

In [ ]:
# Imports and reproducible project paths
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "ames.csv").exists():
    ROOT = ROOT.parent
RAW_PATH = ROOT / "data" / "raw" / "ames.csv"
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {RAW_PATH}. Place ames.csv in data/raw/.")

In [ ]:
df = pd.read_csv(RAW_PATH)

### Visualize dataframe

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["SalePrice"].describe()

# Exploratory Data Analysis — Ames Housing

## 1. Dataset overview
- The dataset contains **2,930 house sales** and **83 columns** (80 features, the target and 2 identifiers, plus an index column added when the CSV was saved).
- **Candidate identifier columns to exclude during modelling:** `Unnamed: 0`, `Order` and `PID` are row/parcel identifiers. They carry no information about the house itself, and keeping them could let a model memorise IDs instead of learning real patterns.
- **Columns with many missing values:** `Pool QC` (2,917 missing), `Misc Feature` (2,824), `Alley` (2,732) and `Fence` (2,358).
  These are most likely **not data errors**: according to the dataset documentation, a missing value here means the house *has no* pool, alley access or fence. They will be encoded as an explicit "None" category rather than imputed.

## 2. Target variable: SalePrice
- **Half of the houses sell between $129,500 and $213,500** (IQR = $84,000), a fairly narrow central range compared with the full range ($12,789 – $755,000).
- **Mean vs. median:** the mean ($180,796) is about $21,000 above the median ($160,000), and the most expensive house is $595,000 above the median while the cheapest is only $147,000 below it. The distribution is **right-skewed**: a small number of expensive houses pull the mean up.
- **Log transform:** taking `log(SalePrice)` makes the distribution close to symmetric. This matters for the model: a linear regression on raw prices would be dominated by the few very expensive houses, and errors would grow with price. On the log scale, errors behave more like *percentage* errors, which is also how a buyer or agent thinks about price.

## 3. Next steps
- Drop the identifier columns.
- Treat "missing = feature absent" columns as an explicit "None" category.
- Model `log(SalePrice)` and convert predictions back to dollars for evaluation.
- Investigate the most expensive and cheapest sales as potential outliers.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["SalePrice"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("SalePrice")

sns.histplot(np.log1p(df["SalePrice"]), bins=50, kde=True, ax=axes[1])
axes[1].set_title("Log-transformed SalePrice")

plt.tight_layout()

plt.savefig(FIGURES_DIR / "saleprice_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

- **Log transform:** `log(SalePrice)` is close to symmetric and centred around 12 (e¹² ≈ $163k, consistent with the median). A short left tail remains: a handful of very cheap sales (below ~$35k) that should be checked as potential outliers. Modelling on the log scale prevents the few very expensive houses from dominating the fit, and errors behave like *percentage* errors, which is how buyers and agents think about price.

### Missing values

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)

missing_table = pd.DataFrame({"missing": missing, "percent": missing_pct})
missing_table

In [ ]:
df.loc[df["Garage Type"].isna(), ["Garage Type", "Garage Area", "Garage Cars"]].head(10) 


In [ ]:
mask = df["Garage Type"].notna() & df["Garage Qual"].isna()
df.loc[mask, ["Garage Type", "Garage Area", "Garage Cars", "Garage Qual", "Garage Yr Blt"]]

**Data-quality decision:** Inspect inconsistent garage records here, but do not delete them during EDA. Any exclusion must be justified and implemented consistently in the modelling pipeline. The preprocessing notebook retains these observations and imputes missing values using training data only.

In [ ]:
garage_cols = ["Garage Type", "Garage Qual", "Garage Cond", "Garage Finish", "Garage Yr Blt", "Garage Area", "Garage Cars"]
df[garage_cols].isna().sum()

**Garage missingness:** Missing garage attributes often indicate no garage. Some records may contain inconsistent or incomplete details; inspect these separately. No rows are removed in this EDA.

In [ ]:
df.loc[df["Pool QC"].isna(), "Pool Area"].value_counts()

In [ ]:
df.loc[df["Bsmt Qual"].isna(), "Total Bsmt SF"].value_counts()

In [ ]:
df["Lot Frontage"].describe()
(df["Lot Frontage"] == 0).sum()

In [ ]:
df.loc[df["Mas Vnr Area"].isna(), ["Mas Vnr Type", "Mas Vnr Area"]]

## 3. Missing values
Missingness is frequently structural (for example, no pool or garage), but not every missing value means absence. We retain the raw data in this notebook and document the missingness patterns. Imputation is fitted only on the training split in notebook 02.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
missing_pct.sort_values().plot.barh(ax=ax)
ax.set_xlabel("Missing values (%)")
ax.set_title("Missing values per column")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "missing_values.png", dpi=150, bbox_inches="tight")
plt.show()

**Preprocessing boundary:** Do not fill missing values in this EDA. Notebook 02 performs imputation after the train/test split to avoid leakage.

In [ ]:
remaining = df.isna().sum()
display(remaining[remaining > 0].sort_values(ascending=False))